# B03 · NumPy 数值计算

> 阶段〇第 3 周。前两周用 Python 列表逐点算信号，本周换成 NumPy：
> 一行代码对一百万个采样点同时运算。这是后续一切（RL 的批量采样、Isaac 的 GPU 并行仿真）的思想源头。

## 学习目标

1. 创建 `ndarray` 并理解 `shape` / `dtype` / `ndim`；
2. 用索引、切片、布尔掩码提取信号片段；
3. 理解**广播（broadcasting）**机制，写出无循环的向量化代码；
4. 用 `timeit` 实测向量化 vs for 循环的性能差距；
5. 使用常用统计函数与线性代数（矩阵乘法、特征值），联系状态空间 $\dot{x} = Ax$ 的稳定性；
6. 用 `default_rng(seed)` 生成可复现的随机数。

## 1. 创建数组：仿真从时间轴开始

`ndarray` 是同类型元素的多维数组。生成仿真时间轴：**推荐 `linspace`（指定点数）**，
`arange`（指定步长）在浮点步长下容易因舍入误差少一个点。

In [1]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
print(a, "| dtype:", a.dtype, "| shape:", a.shape, "| ndim:", a.ndim)

z = np.zeros(5)            # 全 0
o = np.ones((2, 3))        # 全 1（2 行 3 列）
print(z)
print(o)

t_bad = np.arange(0.0, 1.0, 0.1)    # 步长型：点数靠算，浮点有坑
t = np.linspace(0.0, 1.0, 11)       # 点数型：0~1 s 共 11 点 —— 推荐
print("arange:", t_bad)
print("linspace:", t)

[1. 2. 3.] | dtype: float64 | shape: (3,) | ndim: 1
[0. 0. 0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
arange: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
linspace: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]


## 2. 索引、切片与布尔掩码

规则和 Python 列表一致（0 起始、左闭右开、负数倒数），但多了两个利器：
**多维索引** `M[i, j]` 与**布尔掩码** `y[y > 0.9]`——后者是「找超调段、挑异常样本」的标准写法。

In [2]:
t = np.linspace(0, 2, 11)
y = 1 - np.exp(-t / 0.5)          # 一阶环节响应（tau=0.5）
print("y =", np.round(y, 3))

print("首末样本:", y[0], y[-1], "| 第 3~5 个:", np.round(y[2:5], 3))

mask = y > 0.9                     # 布尔数组：每个位置 True/False
print("掩码:", mask)
print("超过 0.9 的样本:", np.round(y[mask], 3))
print("首次超过 0.9 的时刻: t =", t[np.argmax(mask)], "s")   # argmax 找第一个 True

y = [0.    0.33  0.551 0.699 0.798 0.865 0.909 0.939 0.959 0.973 0.982]
首末样本: 0.0 0.9816843611112658 | 第 3~5 个: [0.551 0.699 0.798]
掩码: [False False False False False False  True  True  True  True  True]
超过 0.9 的样本: [0.909 0.939 0.959 0.973 0.982]
首次超过 0.9 的时刻: t = 1.2000000000000002 s


In [3]:
# 二维数组：3 个通道 × 5 个采样点的传感器数据
X = np.array([[1., 2., 3., 4., 5.],
              [10., 20., 30., 40., 50.],
              [0., 0., 0., 1., 1.]])
print("shape:", X.shape)
print("第 1 通道第 2 个采样:", X[1, 2])
print("所有通道的第 0 个采样:", X[:, 0])
print("前两通道、中间三个采样:\n", X[0:2, 1:4])

shape: (3, 5)
第 1 通道第 2 个采样: 30.0
所有通道的第 0 个采样: [ 1. 10.  0.]
前两通道、中间三个采样:
 [[ 2.  3.  4.]
 [20. 30. 40.]]


## 3. 广播机制：不同形状的数组怎么一起算

NumPy 对形状不同的数组做运算时，会**自动「补维」对齐**——标量扩成整段信号、
列向量扩成矩阵。规则：从最后一维往前对齐，维度相等或有一方为 1 即可广播。

In [4]:
t = np.linspace(0, 1, 5)
sig = np.sin(2 * np.pi * t)      # 1 Hz 正弦

print("加直流偏置:", np.round(sig + 1.0, 3))   # 标量广播到每个元素
print("增益 2 倍:", np.round(sig * 2.0, 3))

# 多通道归一化：每行减去自己的均值（行均值扩成与原矩阵同形）
X = np.array([[1., 2., 3., 4., 5.],
              [10., 20., 30., 40., 50.],
              [0., 0., 0., 1., 1.]])
row_mean = X.mean(axis=1, keepdims=True)   # keepdims 保留维度 (3,1) 才能广播
X_centered = X - row_mean
print("去均值后每行均值 ≈ 0:", np.round(X_centered.mean(axis=1), 6))

加直流偏置: [1. 2. 1. 0. 1.]
增益 2 倍: [ 0.  2.  0. -2. -0.]
去均值后每行均值 ≈ 0: [ 0.  0. -0.]


## 4. 向量化 vs for 循环：实测差距

「对整段信号逐点算 sin」——for 循环版本 Python 解释器逐条执行；
向量化版本把循环下沉到 NumPy 底层的 C 代码。用 `timeit` 实测一下。

In [5]:
import timeit

t = np.linspace(0, 10, 1_000_001)   # 一百万个采样点

def loop_sin():
    out = np.empty_like(t)
    for i in range(len(t)):
        out[i] = np.sin(t[i])
    return out

t_vec = timeit.timeit(lambda: np.sin(t), number=5) / 5
t_loop = timeit.timeit(loop_sin, number=1)
print(f"向量化 : {t_vec * 1e3:8.2f} ms/次")
print(f"for 循环: {t_loop * 1e3:8.0f} ms/次")
print(f"加速比 ≈ {t_loop / t_vec:.0f}x")
# 结论：仿真主循环（每个控制周期一次）用 Python 写没问题；
# 但「对一整段采样数据做运算」一定要向量化。

向量化 :     6.21 ms/次
for 循环:      129 ms/次
加速比 ≈ 21x


## 5. 常用统计与随机数

**随机种子**：仿真和 RL 实验必须可复现——用 `np.random.default_rng(seed)` 创建独立随机源，
同一个种子产生完全相同的序列。不要再用旧的 `np.random.seed()` 全局种子。

In [6]:
rng = np.random.default_rng(42)        # 固定种子 → 结果可复现

clean = np.sin(np.linspace(0, 2 * np.pi, 1000))   # 干净的 1 周期正弦
noise = rng.normal(0.0, 0.1, size=1000)           # 均值 0、标准差 0.1 的高斯噪声
meas = clean + noise                               # 模拟传感器读数

print("均值:", round(meas.mean(), 4), "（理论 ≈ 0）")
print("标准差:", round(meas.std(), 4), "（理论 ≈ 0.1）")
print("峰峰值:", round(meas.max() - meas.min(), 3))
print("最大值出现在下标:", meas.argmax(), "/", len(meas))

# 同样的种子再开一个 rng，序列完全一样 —— 这就是「可复现实验」
rng2 = np.random.default_rng(42)
print("复现验证:", np.allclose(rng2.normal(0, 0.1, 1000), noise))

均值: -0.0029 （理论 ≈ 0）
标准差: 0.7158 （理论 ≈ 0.1）
峰峰值: 2.513
最大值出现在下标: 248 / 1000
复现验证: True


## 6. 线性代数：联系现代控制理论

状态空间模型 $\dot{x} = A x + B u$ 中，**$A$ 的特征值决定系统稳定性**：
所有特征值实部 < 0 → 渐近稳定（极点全在左半平面，就是经典控制里的说法）。
NumPy 的 `linalg` 子模块一行就能验证。

In [7]:
# 弹簧-质量-阻尼系统 m=1, k=2, c=3，状态 x = [位置, 速度]
A = np.array([[0.0, 1.0],
              [-2.0, -3.0]])
B = np.array([[0.0],
              [1.0]])

w = np.linalg.eigvals(A)          # 特征值
print("A 的特征值:", w)
print("实部均 < 0 →", np.all(w.real < 0), "（渐近稳定）")

# 矩阵乘法用 @：计算状态导数 dx/dt = A x
x0 = np.array([1.0, 0.0])         # 初始位置 1，速度 0
print("t=0 时刻 dx/dt =", A @ x0)

# 解线性方程组 Ax = -B（求平衡点附近的工作点等场景常用）
x_sol = np.linalg.solve(A, -B.flatten())
print("解 Ax = -B:", x_sol)

# 换个不稳定的 A（c 取负 → 负阻尼）
A_bad = np.array([[0.0, 1.0], [-2.0, 3.0]])
print("负阻尼特征值:", np.linalg.eigvals(A_bad), "→ 实部 > 0，发散")

A 的特征值: [-1. -2.]
实部均 < 0 → True （渐近稳定）
t=0 时刻 dx/dt = [ 0. -2.]
解 Ax = -B: [ 0.5 -0. ]
负阻尼特征值: [1. 2.] → 实部 > 0，发散


## 小结与衔接

- `linspace` 建时间轴；布尔掩码挑样本；广播免去手写循环；
- 向量化比对等 for 循环快 1~2 个数量级——「批量数据运算」永远先想向量化；
- `eigvals(A)` 看稳定性，是自控原理与代码的直接握手；
- `default_rng(seed)` 保证实验可复现——RL 训练对比实验时这是硬性要求。

**下周 B04**：真实数据往往是「带表头的表格 + 需要时间戳」——pandas 管数据，matplotlib 管出图。

---

## ✏️ 练习

> 规则：先独立完成，再点开折叠的参考答案核对。

**练习 1（★，10 分钟，10 分）——进入误差带的时刻**
用 `linspace` 生成 $t \in [0, 5]$ s（1001 点），向量化计算 $y(t) = K(1 - e^{-t/\tau})$（$K=2, \tau=0.5$），
用布尔掩码 + `argmax` 求 $y$ **首次进入并保持在** $\pm 2\%$ 稳态误差带内的时刻
（提示：先构造「尚未进入」的掩码再取反，或直接用 `y >= 0.98*K` 找首个满足的下标——思考两者何时等价）。
**交付物**：打印该时刻并与 $4\tau$ 比较。

**练习 2（★★，20 分钟，20 分）——一次仿真 5 条曲线**
利用广播：构造 `tau = np.array([0.1, 0.2, 0.5, 1.0, 2.0])`（形状 `(5,1)`）与
`t = np.linspace(0, 5, 501)`（形状 `(501,)`），一步算出 `(5, 501)` 的响应矩阵
$Y = 1 - e^{-t/\tau}$。对每条曲线求「达到稳态 95% 所需时间」（逐行用掩码），打印成表格。
**交付物**：响应矩阵的 shape + 5 行结果。

**练习 3（★，10 分钟，10 分）——稳定性判断函数**
写函数 `is_stable(A)`：返回 `np.linalg.eigvals(A)` 实部是否全部 < 0。
测试三个矩阵：`[[0,1],[-2,-3]]`、`[[0,1],[-2,3]]`、`[[0,1],[-1,0]]`（无阻尼，临界稳定——
实部为 0，你的函数会判 False，请用一句话解释工程上为什么这不算「稳定」）。
**交付物**：函数 + 三个测试结果 + 一句解释。

**练习 4（★★，15 分钟，15 分）——蒙特卡洛验证**
用 `default_rng(0)` 生成 100000 个 $\mathcal{N}(0, 0.5^2)$ 样本：
(a) 验证样本标准差 ≈ 0.5；(b) 统计落在 $\pm\sigma$、$\pm 2\sigma$ 内的比例，
对照理论值 68.3% / 95.4%。
**交付物**：打印四个数。

---

<details>
<summary>参考答案（做完再点开）</summary>

**练习 1**：

```python
t = np.linspace(0, 5, 1001)
y = 2 * (1 - np.exp(-t / 0.5))
idx = np.argmax(y >= 0.98 * 2)   # 单调上升 → 首次满足即永久满足
print(t[idx])                    # ≈ 2.0 s ≈ 4τ
```

（一阶环节单调，「首次进入」与「进入并保持」等价；对有超调的二阶系统则必须用「之后不再越界」的写法。）

**练习 2**：

```python
tau = np.array([0.1, 0.2, 0.5, 1.0, 2.0]).reshape(-1, 1)  # (5,1)
t = np.linspace(0, 5, 501)                                 # (501,)
Y = 1 - np.exp(-t / tau)                                   # 广播 → (5,501)
print(Y.shape)
for i, ta in enumerate(tau.flatten()):
    idx = np.argmax(Y[i] >= 0.95)
    print(f"tau={ta}: t95 = {t[idx]:.2f} s (3τ = {3*ta:.2f})")
```

**练习 3**：

```python
def is_stable(A):
    return bool(np.all(np.linalg.eigvals(np.asarray(A, dtype=float)).real < 0))

print(is_stable([[0, 1], [-2, -3]]))   # True
print(is_stable([[0, 1], [-2, 3]]))    # False
print(is_stable([[0, 1], [-1, 0]]))    # False：虚轴上的一对极点
```

无阻尼系统受任意小扰动都会持续等幅振荡、不衰减，工程上无法「停在」平衡点，属临界稳定而非渐近稳定。

**练习 4**：

```python
rng = np.random.default_rng(0)
x = rng.normal(0, 0.5, 100000)
print(x.std())                                       # ≈ 0.5
print(np.mean(np.abs(x) < 0.5))                      # ≈ 0.683
print(np.mean(np.abs(x) < 1.0))                      # ≈ 0.954
```

</details>

---

## 延伸阅读

- [NumPy 官方入门（absolute beginners）](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [NumPy 广播机制图解](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [随机数生成新 API（Generator）说明](https://numpy.org/doc/stable/reference/random/index.html)
- 《利用 Python 进行数据分析》第 4 章（NumPy 基础）